In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from rdkit import Chem
import numpy as np
from tqdm import tqdm

In [ ]:
!pwd

In [ ]:
trainset  = pd.read_csv('./Dataset_train_homo.csv')
trainset

In [ ]:
def calculate_scaffolds(df, smiles_col):
    """
    计算分子骨架 (Scaffold)，返回包含 SMILES 和骨架两列的 DataFrame。
    
    参数：
        df (pd.DataFrame): 包含 SMILES 的数据框。
        smiles_col (str): SMILES 列的名称。
        
    返回：
        pd.DataFrame: 包含原始 SMILES 和对应骨架的 DataFrame。
    """
    scaffolds = []
    for smiles in df[smiles_col]:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
            else:
                scaffold = None
        except Exception as e:
            scaffold = None
        scaffolds.append(scaffold)

    result_df = pd.DataFrame({smiles_col: df[smiles_col], "scaffold": scaffolds})
    return result_df

In [ ]:
trainset_scaffold = calculate_scaffolds(trainset, 'SMILES')

In [ ]:
trainset_scaffold.head()

In [ ]:
def calculate_molecular_properties(df, smiles_col):
    """
    计算分子的物理化学性质，包括分子量、原子数、LogP、氢键受体数量、
    氢键供体数量、旋转键数量和环的数量。
    
    参数：
        df (pd.DataFrame): 包含分子 SMILES 的 DataFrame。
        smiles_col (str): SMILES 列名。
        
    返回：
        pd.DataFrame: 包含原始列和新计算性质列的 DataFrame。
    """
    # 定义空列表存储计算结果
    molecular_weights = []
    atom_counts = []
    logp_values = []
    h_acceptors = []
    h_donors = []
    rotatable_bonds = []
    ring_counts = []

    for smiles in df[smiles_col]:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                # 计算性质
                molecular_weights.append(Descriptors.MolWt(mol))
                atom_counts.append(mol.GetNumAtoms())
                logp_values.append(Descriptors.MolLogP(mol))
                h_acceptors.append(Descriptors.NumHAcceptors(mol))
                h_donors.append(Descriptors.NumHDonors(mol))
                rotatable_bonds.append(Descriptors.NumRotatableBonds(mol))
                ring_counts.append(Descriptors.RingCount(mol))
            else:
                # 如果解析失败，填充空值
                molecular_weights.append(None)
                atom_counts.append(None)
                logp_values.append(None)
                h_acceptors.append(None)
                h_donors.append(None)
                rotatable_bonds.append(None)
                ring_counts.append(None)
        except Exception as e:
            # 捕获异常并填充空值
            molecular_weights.append(None)
            atom_counts.append(None)
            logp_values.append(None)
            h_acceptors.append(None)
            h_donors.append(None)
            rotatable_bonds.append(None)
            ring_counts.append(None)

    # 将新列加入原始 DataFrame
    df['molecular_weight'] = molecular_weights
    df['atom_count'] = atom_counts
    df['logP'] = logp_values
    df['h_acceptors'] = h_acceptors
    df['h_donors'] = h_donors
    df['rotatable_bonds'] = rotatable_bonds
    df['ring_count'] = ring_counts

    return df

In [ ]:
trainset_prop = calculate_molecular_properties(trainset_scaffold, 'SMILES')

In [ ]:
trainset_prop.columns = trainset_prop.columns.str.lower()
trainset_prop.head()

In [ ]:
samset = pd.read_csv('./sam.csv')

In [ ]:
samset.head()

In [ ]:
samset_prop = calculate_molecular_properties(samset, 'smiles')

In [ ]:
samset_prop.head()

In [ ]:
samset_sca_smiles = samset_prop['scaffold_smiles']
trainset_sca_smiles = trainset_prop['scaffold']

In [ ]:
class utils():
    def __init__(self):
        super().__init__()
        self.mol_from_smiles=Chem.MolFromSmiles
        self.fingerprint=Chem.AllChem.GetMorganFingerprintAsBitVect
        self.similarity = DataStructs.FingerprintSimilarity
    
    def mol(self,x):
        mol=self.mol_from_smiles(x)
        return mol

    def ECFP(self,mol):
        ecfp=self.fingerprint(mol, radius=2, nBits=2048)
        return ecfp 

In [ ]:
samset_sca_mol=[utils().mol(i) for i in samset_sca_smiles]
trainset_sca_mol=[utils().mol(i) for i in trainset_sca_smiles]
samset_sca_ecfp=[utils().ECFP(i) for  i in samset_sca_mol]
trainset_sca_ecfp=[utils().ECFP(i) for i in trainset_sca_mol]

In [ ]:
# calculate the similarity matrices of set 1 and set 2
trainset_similarities=np.zeros((len(trainset_sca_smiles),len(samset_sca_smiles)))
for i in tqdm(range(len(trainset_sca_ecfp)), desc="Calculating similarities", unit="batch"):
    for j in range(len(samset_sca_ecfp)):
        score=DataStructs.FingerprintSimilarity(trainset_sca_ecfp[i],samset_sca_ecfp[j])
        trainset_similarities[i,j]=score

In [ ]:
trainset_similarities.shape

In [ ]:
trainset_prop['average_similarity'] = np.sum(trainset_similarities, axis=1) / len(samset_sca_smiles)
trainset_prop['max_similarity'] = np.max(trainset_similarities, axis=1)
trainset_prop.head()

In [ ]:
# calculate the similarity matrices of gen set and lit set
sam_similarities=np.zeros((len(samset_sca_smiles),len(samset_sca_smiles)))
for i in tqdm(range(len(samset_sca_ecfp)), desc="Calculating similarities", unit="batch"):
    for j in range(len(samset_sca_ecfp)):
        score=DataStructs.FingerprintSimilarity(samset_sca_ecfp[i],samset_sca_ecfp[j])
        sam_similarities[i,j]=score

In [ ]:
sam_similarities.shape

In [ ]:
samset_prop['average_similarity'] = np.sum(sam_similarities, axis=1) / len(samset_sca_smiles)
samset_prop['max_similarity'] = np.max(sam_similarities, axis=1)
samset_prop.head()

In [ ]:
samset_prop.columns = samset_prop.columns.str.lower()

In [ ]:
samset_prop.to_csv("./samset_prop.csv")

In [ ]:
trainset_prop.to_csv("./trainset_prop.csv")

In [ ]:
print("Trainset columns:", trainset_prop.columns)
print("Samset columns:", samset_prop.columns)


In [ ]:
import matplotlib
print(matplotlib.rcParams["font.family"])


In [ ]:
def plot_property_histograms_double_y(trainset, samset, properties):
    # 设置字体和样式
    plt.rcParams["font.family"] = "Arial"
    plt.rcParams["font.size"] = 18
    plt.rcParams["axes.linewidth"] = 2

    # 创建子图
    n_properties = len(properties)
    n_rows = (n_properties + 1) // 2  # 每行放两个图
    fig, axes = plt.subplots(n_rows, 2, figsize=(18, 8 * n_rows), dpi=600)
    axes = axes.flatten()

    for i, prop in enumerate(properties):
        if prop not in trainset.columns or prop not in samset.columns:
            print(f"Property {prop} not found in one of the datasets. Skipping...")
            continue
        
        # 数据
        train_values = trainset[prop].dropna()
        sam_values = samset[prop].dropna()
        
        # 计算直方图
        bins = np.histogram_bin_edges(np.concatenate([train_values, sam_values]), bins='auto')

        # 绘制直方图
        ax = axes[i]
        ax2 = ax.twinx()  # 创建右侧 y 轴

        ax.hist(train_values, bins=bins, alpha=0.7, color='b', label="Trainset", edgecolor='black')
        ax2.hist(sam_values, bins=bins, alpha=0.7, color='r', label="Samset", edgecolor='black')

        # 设置标题、轴标签和图例
        ax.set_title(f"{prop.replace('_', ' ').capitalize()} Distribution", fontweight="bold")
        ax.set_xlabel(prop.replace("_", " ").capitalize(), fontweight="bold")
        ax.set_ylabel("Counts (Trainset)", color='b', fontweight="bold")
        ax2.set_ylabel("Counts (Samset)", color='r', fontweight="bold")

        # 设置刻度颜色
        ax.tick_params(axis='y', colors='b', width=2)
        ax2.tick_params(axis='y', colors='r', width=2)
        ax.tick_params(axis='x', width=2)
        ax.spines['top'].set_linewidth(2)
        ax.spines['left'].set_linewidth(2)
        ax.spines['bottom'].set_linewidth(2)
        ax2.spines['right'].set_linewidth(2)

        # 图例
        ax.legend(loc="upper left", frameon=False)
        ax2.legend(loc="upper right", frameon=False)
    
    # 移除多余的子图
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # 调整布局并显示
    plt.tight_layout()
    plt.show()

# 调用函数
properties = ["molecular_weight", "atom_count", "logp", "h_acceptors", "h_donors", "rotatable_bonds", "ring_count", "average_similarity"]
plot_property_histograms_double_y(trainset_prop, samset_prop, properties)